For each session:
- plot PA with and without subsampling
- some sort of test for distribution similarity of real and predicted 

In [ ]:
from behave_analysis.process.session import get_experiment 
from JR_test_scripts.tracking2features import tracking_to_features, extract_pa_across_sesh

import os
import dill as pickle
import numpy as np
import matplotlib.pyplot as plt
import re

In [ ]:
"""Run it for barrier sessions!"""

from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept, JAL3_22aug

from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_11thSept, JAL4_28aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr, JAL7_30apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip3_7may, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may, JAL8_tiny_3may, JAL8_21may

experiments_objects = [JAL3_7sept, JAL3_4sept, JAL3_1sept, JAL3_25aug, JAL3_22aug,
                        JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept,
                        JAL005_8thSept, JAL005_21stSept, # JAL005_5thSept this one doesn't flip, but can be used as first barrier appearance
                        JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr,
                        JAL7_sesh8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_sesh9_16apr, JAL7_23apr,
                        JAL8_flip1_25apr,JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_flip4_10may, JAL8_14may
                        JAL8_tiny_3may, JAL8_21may, JAL7_30apr]

# fixed vars
ceph_path = r"Z:\Jasmine_Laurence\Experimental_Data"
winstor_path = r"Y:\Laurence"
save_path = r"Z:\Jasmine_Laurence\LDA_overview"
conditions = ['shelter_only','barrier_pre_flip','barrier_post_flip']
all_angles = ['hdir','hsa','h_preflipbar_a','h_postflipbar_a','h_rightbar_a','h_leftbar_a']
all_features = ['shelter','preflip_barrier','postflip_barrier','left_barrier', 'right_barrier', 'arena_bottom', 'arena_top']

In [ ]:
"""Run it for shelter sessions!"""

from behave_analysis.database.Experiments.JAL003_ex import JAL3_17aug

from behave_analysis.database.Experiments.JAL004_ex import JAL4_17aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_2ndSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_shelt_4mar      

from behave_analysis.database.Experiments.JAL007_ex import JAL7_empty_shelter_5mar

from behave_analysis.database.Experiments.JAL008_ex import JAL8_shelt_22apr

experiments_objects = [JAL005_2ndSept,JAL4_17aug, JAL3_17aug, JAL8_shelt_22apr, JAL7_empty_shelter_5mar, JAL6_shelt_4mar]#, JAL005_5thSept]

# fixed vars
ceph_path = r"Z:\Jasmine_Laurence\Experimental_Data"
winstor_path = r"Y:\Laurence"
save_path = r"Z:\Jasmine_Laurence\LDA_overview"
conditions = ['pre_shelter',"shelter_present"]
all_angles = ['hdir','hsa','h_preflipbar_a','h_postflipbar_a','h_rightbar_a','h_leftbar_a']
all_features = ['shelter','preflip_barrier','postflip_barrier','left_barrier', 'right_barrier', 'arena_bottom', 'arena_top']

In [ ]:
settings = ['all_angles_fr','all_angles_fr_excl_prox_15cm','all_angles_subsampled_fr','all_angles_subsampled_fr_excl_prox_15cm']
colorz = plt.cm.jet(np.linspace(0, 1, len(settings)))

# iterate over sessions
for s,sesh in enumerate(experiments_objects):
    # load session deets
    session = get_experiment(sesh)
    fig, axs = plt.subplots(1,len(conditions))
    plt.rcParams['figure.figsize'] = [15, 5]
    line_object = []
    for idx,c in enumerate(conditions):
        for idx_s, set in enumerate(settings):
            coef_path = os.path.join(session.base_path,
                                        session.processed_path,
                                        'models',
                                        'LDA',
                                        set,
                                        r'good\experimental_conditions\all',
                                        c,
                                        str('good_' + c + '_LDA_pa.pkl'))
            with open(coef_path, "rb") as dill_file:
                coef = pickle.load(dill_file)
            angles = [key for key, val in coef.items() if np.logical_and(not re.search("time", key),not re.search("randP", key))]
            pa = [val for key, val in coef.items() if np.logical_and(not re.search("time", key),not re.search("randP", key))]
            line, = axs[idx].plot(angles,pa, color = colorz[idx_s])
            if idx == 0: line_object.append(line)
            x = [len(angles)]
            pa = [val for key, val in coef.items() if np.logical_and(not re.search("time", key),re.search("randP", key))]
            violin = axs[idx].violinplot(dataset = pa,positions = x,showextrema = False)
            for v in violin['bodies']:
                v.set_alpha(.5)
                v.set_facecolor(colorz[idx_s])
            axs[idx].set_ylim([0,1])
            axs[idx].set_title(c)
            axs[idx].set_xticks(np.arange(len(angles)))
            axs[idx].set_xticklabels(angles,rotation = 45, ha="right")
    axs[idx].legend(line_object,settings,
                    loc='center left', 
                    bbox_to_anchor=(1.5, .5),
                    columnspacing=1.0, labelspacing=0.2,
                    handletextpad=0.5, handlelength=1.5)
    axs[0].set_ylabel('prediction accuracy')

    # save image to folder
    name = sesh.nick_name + '_' + sesh.experiment_date
    save_path = r"Z:\Jasmine_Laurence\LDA_overview"
    plt.savefig(save_path+'/'+name+'.png', bbox_inches='tight')
    plt.close()